In [ ]:
import subprocess, sys, os, time, json, gc
os.environ.setdefault("WIDTHS", "160"); os.environ.setdefault("MC_SAMPLES", "2e9")
print(subprocess.run("git clone -q https://github.com/alignment-research-center/mlp_cumulant_propagation.git /kaggle/working/mcp && pip install -q jaxtyping einops opt_einsum 2>&1 | tail -1", shell=True, capture_output=True, text=True).stdout)
sys.path.insert(0, "/kaggle/working/mcp/src")
import torch, numpy as np
print(torch.__version__, "cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
from mlp_kprop.mlp import MLP
from mlp_kprop.kprop_harmonic import mlp_kprop, Kind
from mlp_kprop.flop_utils import NamedFlopCounter
torch.set_grad_enabled(False)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
N_MLPS = int(os.environ.get("N_MLPS", "3"))
MODE = os.environ.get("MODE", "synthetic")   # "synthetic": He-init width x depth MLPs with a GPU Monte-Carlo truth; "hf": public dataset
WIDTH = int(os.environ.get("WIDTH", "96")); DEPTH = int(os.environ.get("DEPTH", "16"))
MC_SAMPLES = int(float(os.environ.get("MC_SAMPLES", "2e9")))
rows = []
if MODE == "hf":
    from datasets import load_dataset
    REV = os.environ.get("REV", "v1-phase1")
    ds = load_dataset("aicrowd/arc-whestbench-public-2026", revision=REV, split="mini").with_format("numpy")
    for i in range(N_MLPS):
        r = ds[i]
        rows.append({"mlp_name": str(r["mlp_name"]), "weights": r["weights"][:DEPTH], "final_means": r["all_layer_means"][DEPTH - 1]})
else:
    def mc_truth(W, n_samples, chunk=1 << 16, seed=0):
        """whestbench convention h_l = relu(h_{l-1} @ W_l), x ~ N(0, I); float32 forward, float64 accumulation."""
        Wt = [torch.as_tensor(w, device=dev, dtype=torch.float32) for w in W]
        g = torch.Generator(device=dev); g.manual_seed(seed)
        acc = torch.zeros(W.shape[1], device=dev, dtype=torch.float64); done = 0
        while done < n_samples:
            m = min(chunk, n_samples - done)
            h = torch.randn(m, W.shape[1], device=dev, generator=g)
            for w in Wt: h = torch.relu(h @ w)
            acc += h.sum(0, dtype=torch.float64); done += m
        return (acc / done).cpu().numpy()
    WIDTHS = [int(w) for w in os.environ.get("WIDTHS", str(WIDTH)).split(",")]
    for WIDTH_ in WIDTHS:
        for i in range(N_MLPS):
            rng = np.random.default_rng(1000 + i)
            W = (rng.standard_normal((DEPTH, WIDTH_, WIDTH_)) * np.sqrt(2.0 / WIDTH_)).astype(np.float32)
            t0 = time.time(); truth = mc_truth(W, MC_SAMPLES, seed=i)
            print(f"synthetic-{WIDTH_}-{i}: MC truth {MC_SAMPLES:.0e} samples in {time.time()-t0:.0f}s, mean {truth.mean():.4f}", flush=True)
            rows.append({"mlp_name": f"synth{WIDTH_}x{DEPTH}-{i}", "weights": W, "final_means": truth})
print([r["mlp_name"] for r in rows], rows[0]["weights"].shape, rows[0]["weights"].dtype, "truth", rows[0]["final_means"].shape)
np.savez("/kaggle/working/mlps_used.npz", **{f"{r['mlp_name']}_W": r["weights"] for r in rows}, **{f"{r['mlp_name']}_truth": r["final_means"] for r in rows})


In [ ]:
def build_mlp(W, dtype):
    """W: (16, 1024, 1024) float32, whestbench convention h_l = relu(h_{l-1} @ W_l). Torch Linear computes x @ weight.T -> weight = W_l.T.
    17 linear layers so that act15 (= post-ReLU of layer 16) exists; the 17th layer is never consumed with up_to_layer='act15'."""
    n = W.shape[1]
    L = W.shape[0]
    mlp = MLP(input_dim=n, hidden_dim=n, output_dim=n, num_layers=L + 1, nonlin="relu").to(dev).to(dtype)
    with torch.no_grad():
        for i in range(L):
            mlp.Ws[i].weight.copy_(torch.as_tensor(W[i].T, dtype=dtype, device=dev))
        mlp.Ws[L].weight.copy_(torch.eye(n, dtype=dtype, device=dev))
    return mlp

def run(mlp, k_max, kind, factor, dtype, device):
    n = mlp.input_dim
    mlp = mlp.to(device)
    K_in = {1: torch.zeros(n, device=device, dtype=dtype), 2: torch.eye(n, device=device, dtype=dtype)}
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    with NamedFlopCounter() as counter:
        out = mlp_kprop(mlp, K_in, k_max=k_max, kind=kind, factor=factor, output_all=True, up_to_layer=f"act{len(mlp.Ws) - 2}")
    if device.type == "cuda": torch.cuda.synchronize()
    mean = out["act15"][1].to_tensor().detach().cpu().double().numpy()
    flops = counter.mode.get_total_flops(); secs = time.time() - t0
    bd = {"raw_total": flops, "factored_total": counter.total(),
          "factored": {k: int(v) for k, v in counter.flop_dict().items()},
          "raw": {k: int(v) for k, v in counter.raw_flop_dict().items()}}
    del out, K_in; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return mean, flops, secs, bd

def run_any(mlp, k, kind, fac, dtype):
    """GPU only. A CUDA OOM is reported and the configuration skipped (a CPU fallback exhausts host RAM and kills the kernel)."""
    try:
        return run(mlp, k, kind, fac, dtype, dev) + (dev.type,)
    except torch.cuda.OutOfMemoryError as e:
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError("cuda OOM: " + str(e)[:120])

CONFIGS = [("k3_simple_factored", 3, Kind.SIMPLE, True), ("k4_simple_dense", 4, Kind.SIMPLE, False)]
SKIP = set(os.environ.get("SKIP", "").split(",")) - {""}
DT = {"f32": torch.float32}
results = {}
# smoke test: every configuration once at width 32 (seconds) so that a code error shows up before the long MC truth is spent
_W = (np.random.default_rng(0).standard_normal((DEPTH, 32, 32)) * np.sqrt(2.0 / 32)).astype(np.float32)
_mlp = build_mlp(_W, torch.float32)
for tag, k, kind, fac in CONFIGS:
    try:
        _m, _f, _s, _bd, _w = run_any(_mlp, k, kind, fac, torch.float32); print(f"smoke {tag}: ok, {_f/2**31:.3f} u", flush=True)
    except Exception as e:
        print(f"smoke {tag}: FAILED {repr(e)[:200]}", flush=True)
del _mlp; gc.collect(); torch.cuda.empty_cache()
for r in rows:
    truth = r["final_means"].astype(np.float64); W = r["weights"]
    for dname, dtype in DT.items():
        torch.set_default_dtype(dtype)
        mlp = build_mlp(W, dtype)
        for tag, k, kind, fac in CONFIGS:
            if tag in SKIP: continue
            key = f"{r['mlp_name']}|{tag}|{dname}"
            try:
                mean, flops, secs, bd, where = run_any(mlp, k, kind, fac, dtype)
                mse = float(np.mean((mean - truth) ** 2)); units = flops / 2**31; fu = bd["factored_total"] / 2**31
                results[key] = {"mse": mse, "flops": flops, "units": units, "util_B": flops / 2**41, "factored_units": fu,
                                "secs": secs, "device": where, "breakdown": bd}
                print(f"{key}: raw MSE {mse:.3e}  raw FLOPs {units:.0f} u ({flops/2**41:.3f} B)  factored {fu:.0f} u ({fu/1024:.3f} B)  {secs:.0f}s on {where}", flush=True)
                for kk, vv in sorted(bd["factored"].items(), key=lambda t: -t[1])[:12]:
                    print(f"      {vv/2**31:8.1f} u  {kk}", flush=True)
            except Exception as e:
                results[key] = {"error": repr(e)[:400]}; print(key, "FAILED:", repr(e)[:400], flush=True)
            finally:
                gc.collect()
                if dev.type == "cuda": torch.cuda.empty_cache()
            json.dump(results, open("/kaggle/working/kprop_results.json", "w"), indent=1)